# KhataCall: serve Gnani Evon v3.3 on Kaggle

Runs Evon v3.3 (30B MoE, `nemotron_h`) with **llama.cpp** on Kaggle's two T4 GPUs
and exposes it through a **Cloudflare quick tunnel** as an OpenAI-compatible API.
KhataCall's `app/adapters/evon_llm.py` talks to it unchanged.

**Before you press Run All**, in the right-hand panel:
1. **Accelerator: GPU T4 x2** (not P100 - one 16 GB card is too small)
2. **Internet: On** (needs a phone-verified Kaggle account)

The first run takes roughly **20-30 minutes**: ~15 to build llama.cpp for the
T4, ~5 to download the 25 GB model (the two overlap), ~2 to load it. At the end,
step 8 prints three lines to paste into your local `.env`.

Why llama.cpp and not vLLM: the official BF16 weights are ~60 GB and the two
T4s have 32 GB between them. A 4-bit GGUF fits. See DECISIONS.md D-027.

In [ ]:
# 1. What do we have? Two T4s, and where is the disk space.
import os, shutil, subprocess, sys, time, json, re, secrets, urllib.request

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

# /kaggle/working is capped at 20 GB, too small for a 25 GB model. Use the
# largest scratch directory instead. It is not saved after the session, which is fine.
candidates = [d for d in ("/kaggle/tmp", "/tmp", "/root") if os.path.isdir(d)]
for d in candidates + ["/kaggle/working"]:
    print(f"{d:16} {shutil.disk_usage(d).free / 1e9:6.1f} GB free")
WORK = max(candidates, key=lambda d: shutil.disk_usage(d).free) + "/evon"
os.makedirs(WORK, exist_ok=True)
os.environ["WORK"] = WORK
print("\nusing", WORK)

In [ ]:
# 2. Settings. Change QUANT if the model does not fit (see step 6).
REPO = "luminenio/gnani-evon-v3.3-30B-A3B-GGUF"  # community GGUF of gnani/gnani-evon-v3.3-30B-A3B
QUANT = "Q4_K_M"  # 24.7 GB. If loading runs out of GPU memory, use "IQ4_XS" (18.4 GB).
SIZE_GB = {"Q4_K_M": 24.7, "Q4_K_S": 22.1, "IQ4_XS": 18.4, "Q3_K_M": 20.0}.get(QUANT, 0)
FILE = f"gnani-evon-v3.3-30B-A3B-{QUANT}.gguf"
MODEL_PATH = f"{WORK}/{FILE}"
ALIAS = "gnani-evon-v3.3"  # what to put in EVON_MODEL
CTX = 8192                 # KhataCall's longest prompt is a few thousand tokens
PORT = 8080

# The tunnel URL is public. This key is what stops anyone else using your GPUs.
API_KEY = secrets.token_urlsafe(24)
print("settings ready:", FILE)

In [ ]:
# 3. Start the model download in the background. It runs while llama.cpp builds.
# Use Kaggle's preinstalled huggingface_hub as-is. Upgrading it to 2.x breaks the
# preinstalled transformers, datasets and gradio, and 2.x dropped the old
# hf_transfer extra anyway; the bundled version already downloads quickly.
download = subprocess.Popen(
    [sys.executable, "-c",
     "import huggingface_hub as h; print('huggingface_hub', h.__version__, flush=True);"
     f"h.hf_hub_download({REPO!r}, {FILE!r}, local_dir={WORK!r}); print('done', flush=True)"],
    stdout=open(f"{WORK}/download.log", "w"), stderr=subprocess.STDOUT,
)
print("downloading", FILE, "in the background, pid", download.pid)

In [ ]:
%%bash
# 4. Build llama.cpp with CUDA, for the T4 only (compute capability 7.5).
#
# CMAKE_CUDA_ARCHITECTURES=75: real T4 machine code, and one architecture instead
#   of a dozen, so the build is several times faster. (The prebuilt release binary
#   carries the T4 only as 75-virtual PTX, which the driver would JIT-compile on
#   first use in every session - D-027.)
# GGML_CUDA_NO_VMM=ON: Kaggle's CUDA toolkit has no libcuda.so for CMake to link
#   ("CUDA::cuda_driver ... target was not found"). Only VMM, a memory-pool
#   optimisation, needs it; ggml falls back to its ordinary pool allocator.
set -e
cd "$WORK"
[ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp
cd llama.cpp
if [ -x build/bin/llama-server ]; then
    echo "already built this session, skipping"
    exit 0
fi
nvcc --version | tail -1
cmake --version | head -1 || pip install -q cmake

rm -rf build
if ! cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_CUDA_NO_VMM=ON         -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > cmake.log 2>&1; then
    tail -40 cmake.log
    exit 1
fi

start=$(date +%s)
cmake --build build --target llama-server -j 4 > build.log 2>&1 &
pid=$!
while kill -0 "$pid" 2>/dev/null; do
    sleep 60
    step=$(grep -oE '\[ *[0-9]+%\]|\[[0-9]+/[0-9]+\]' build.log | tail -1)
    echo "  building... $step  ($(( ($(date +%s) - start) / 60 )) min)"
done
if ! wait "$pid"; then
    grep -E "error" build.log | head -20
    tail -20 build.log
    exit 1
fi
echo "built in $(( ($(date +%s) - start) / 60 )) min"
ls -la build/bin/llama-server

In [ ]:
# 5. Wait for the download to finish.
def downloaded_gb():
    # The file only appears at MODEL_PATH when complete; until then the bytes sit
    # in a *.incomplete file under WORK/.cache, so count those too.
    total = 0
    for root, _, files in os.walk(WORK):
        if "llama.cpp" in root:
            continue
        total += sum(os.path.getsize(os.path.join(root, f))
                     for f in files if f.endswith((".gguf", ".incomplete")))
    return total / 1e9

while download.poll() is None:
    print(f"\r  {downloaded_gb():5.1f} of ~{SIZE_GB} GB", end="", flush=True)
    time.sleep(10)
print()
assert download.returncode == 0, open(f"{WORK}/download.log").read()[-2000:]
print("downloaded", MODEL_PATH, f"{os.path.getsize(MODEL_PATH) / 1e9:.1f} GB")

In [ ]:
# 6. Start llama-server across both GPUs.
#   -ngl 99    every layer on the GPUs (split across both by default)
#   --jinja    use the model's own chat template; REQUIRED for tool calls
#   --api-key  the key from step 2
server = subprocess.Popen(
    [f"{WORK}/llama.cpp/build/bin/llama-server",
     "-m", MODEL_PATH, "--alias", ALIAS,
     "--host", "127.0.0.1", "--port", str(PORT),
     "-ngl", "99", "-c", str(CTX), "--parallel", "1",
     "--jinja", "--api-key", API_KEY],
    stdout=open(f"{WORK}/server.log", "w"), stderr=subprocess.STDOUT,
)

for _ in range(120):
    if server.poll() is not None:
        break
    try:
        if urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5).status == 200:
            break
    except Exception:
        pass
    time.sleep(5)

log = open(f"{WORK}/server.log").read()
if server.poll() is not None or "out of memory" in log.lower():
    print(log[-3000:])
    raise SystemExit("llama-server did not start. If it ran out of memory, set "
                     "QUANT = 'IQ4_XS' in step 2 and re-run from step 2.")
print("llama-server is up")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 7. One tool call, locally, before exposing anything.
body = {
    "model": ALIAS,
    "messages": [
        {"role": "system", "content": "Tum ek kirana dukaan ke assistant ho. Amount hamesha shabdon mein do."},
        {"role": "user", "content": "Sharma ji ka dedh sau udhaar likho"},
    ],
    "tools": [{"type": "function", "function": {
        "name": "add_credit", "description": "Record credit against a customer",
        "parameters": {"type": "object", "properties": {
            "customer_name": {"type": "string"}, "amount_spoken": {"type": "string"}},
            "required": ["customer_name", "amount_spoken"]}}}],
    "chat_template_kwargs": {"enable_thinking": False},
    "max_tokens": 200, "temperature": 0.2,
}
request = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/v1/chat/completions", data=json.dumps(body).encode(),
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
start = time.time()
reply = json.load(urllib.request.urlopen(request, timeout=180))
message = reply["choices"][0]["message"]
print(f"{time.time() - start:.1f} s")
print("tool_calls:", json.dumps(message.get("tool_calls"), ensure_ascii=False, indent=1))
print("content:", repr(message.get("content")))
# Good: one add_credit call with amount_spoken "dedh sau", and empty content.
# No tool_calls but JSON in content: run evon_check with --tool-mode prompt.

In [ ]:
# 8. Expose it through a Cloudflare quick tunnel and print the .env lines.
if not os.path.exists(f"{WORK}/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        f"{WORK}/cloudflared")
    os.chmod(f"{WORK}/cloudflared", 0o755)

tunnel = subprocess.Popen(
    [f"{WORK}/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=open(f"{WORK}/tunnel.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{WORK}/tunnel.log").read())
    if found:
        url = found.group(0)
        break
    time.sleep(2)
assert url, open(f"{WORK}/tunnel.log").read()[-2000:]

print("Paste these into your local .env. They change every session.\n")
print(f"EVON_BASE_URL={url}/v1")
print(f"EVON_API_KEY={API_KEY}")
print(f"EVON_MODEL={ALIAS}")
print("\nThen run:  uv run python -m scripts.evon_check")

In [ ]:
# 9. Keep this cell running while you use Evon. It keeps the session active and
# shows the server is alive. Interrupt it, then Stop Session, when you are done:
# the free quota is 30 GPU-hours a week and an idle server still counts.
while True:
    try:
        ok = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5).status == 200
    except Exception:
        ok = False
    last = open(f"{WORK}/server.log").read().strip().splitlines()[-1:] or [""]
    print(time.strftime("%H:%M:%S"), "healthy" if ok else "DOWN", "|", last[0][:100], flush=True)
    time.sleep(60)